# Qdrant Vector DB In-Depth Playground
This notebook explores connection initialization, creating collections, embedding generation, point upserts, and payload filtering using the configured Qdrant settings.

In [1]:
import sys
from pathlib import Path

# Locate and append project root
def get_project_root() -> Path:
    current = Path.cwd()
    for _ in range(5):
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    return Path.cwd()

sys.path.append(str(get_project_root().resolve()))

In [2]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue
import httpx
import uuid
from src.ragforge.config import QDRANT_URL, OLLAMA_URL, DEFAULT_EMBEDDING_MODEL, DEFAULT_COLLECTION

print(f"Configured Qdrant URL: {QDRANT_URL}")
print(f"Configured Ollama URL: {OLLAMA_URL}")
print(f"Default Collection: {DEFAULT_COLLECTION}")

Configured Qdrant URL: http://localhost:6333
Configured Ollama URL: http://localhost:11434
Default Collection: ragforge-collection


## 1. Connect and Create Collection

In [3]:
client = QdrantClient(url=QDRANT_URL)
test_collection = "notebook-exploration-collection"

# Clean up existing
if client.collection_exists(test_collection):
    client.delete_collection(test_collection)
    print(f"Deleted existing collection '{test_collection}'")

# Create new collection matching nomic-embed-text size (768)
client.create_collection(
    collection_name=test_collection,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)
print(f"Created collection '{test_collection}' successfully!")

Created collection 'notebook-exploration-collection' successfully!


## 2. Generate Embeddings & Upsert Points with Session Tagging

In [4]:
def embed_text(text: str) -> list[float]:
    response = httpx.post(
        f"{OLLAMA_URL}/api/embeddings",
        json={"model": DEFAULT_EMBEDDING_MODEL, "prompt": text},
        timeout=15.0
    )
    response.raise_for_status()
    return response.json()["embedding"]

session_a = "session-111-aaa"
session_b = "session-222-bbb"

docs = [
    {"id": str(uuid.uuid4()), "text": "Temporal manages distributed workflow orchestration state.", "session_id": session_a},
    {"id": str(uuid.uuid4()), "text": "Qdrant stores high-dimensional dense vector embeddings.", "session_id": session_a},
    {"id": str(uuid.uuid4()), "text": "OpenProject is an enterprise project management platform.", "session_id": session_b}
]

points = []
for doc in docs:
    vec = embed_text(doc["text"])
    points.append(PointStruct(
        id=doc["id"],
        vector=vec,
        payload={"text": doc["text"], "session_id": doc["session_id"]}
    ))

client.upsert(collection_name=test_collection, points=points)
print(f"Upserted {len(points)} documents into Qdrant!")

Upserted 3 documents into Qdrant!


## 3. Querying & Filtering by Session ID

In [6]:
query = "How to store embeddings?"
query_vector = embed_text(query)

# Search A: Filtered to Session A
res_a = client.query_points(
    collection_name=test_collection,
    query=query_vector,
    query_filter=Filter(
        must=[FieldCondition(key="session_id", match=MatchValue(value=session_a))]
    ),
    limit=2
)

print("=== Session A Search Results ===")
for hit in res_a.points:
    print(f"[Score: {hit.score:.4f}] {hit.payload['text']} (Session: {hit.payload['session_id']})")

# Search B: Filtered to Session B
res_b = client.query_points(
    collection_name=test_collection,
    query=query_vector,
    query_filter=Filter(
        must=[FieldCondition(key="session_id", match=MatchValue(value=session_b))]
    ),
    limit=2
)

print("=== Session B Search Results ===")
for hit in res_b.points:
    print(f"[Score: {hit.score:.4f}] {hit.payload['text']} (Session: {hit.payload['session_id']})")

=== Session A Search Results ===
[Score: 0.7160] Qdrant stores high-dimensional dense vector embeddings. (Session: session-111-aaa)
[Score: 0.3555] Temporal manages distributed workflow orchestration state. (Session: session-111-aaa)
=== Session B Search Results ===
[Score: 0.2627] OpenProject is an enterprise project management platform. (Session: session-222-bbb)


## 4. Clean up

In [7]:
client.delete_collection(test_collection)
print("Deleted temporary collection successfully!")

Deleted temporary collection successfully!
